# 11 · GSE232381 · bulk_RNA_seq · metadata

Reads the series matrix and the column headers of the authors' Excel file. Writes
`data/run_artifacts/GSE232381/metadata.rds` and `authors_columns.rds`.

GEO records one clinical field: active or inactive lupus nephritis (the field `treatment`).
**GEO records no age, sex or SLEDAI.** Only the GEO accession and the activity label are kept.

**Personal information.** In GEO, three of the 25 sample descriptions (`!Sample_description`) begin with
what appear to be personal names instead of numeric sample codes, and the authors' Excel file uses the same
labels in its column headers. This notebook never prints a description or a header: it matches each Excel
column to its GSM identifier and saves only the identifiers.

## Read the sample characteristics

In [1]:
source("../src/paths.R")
suppressMessages({library(GEOquery); library(Biobase); library(readxl)})
pd <- pData(suppressMessages(getGEO(filename = raw("GSE232381", "GSE232381_series_matrix.txt.gz"), getGPL = FALSE)))
grep(":ch1$", names(pd), value = TRUE)

[1] "cell type:ch1" "time:ch1"      "treatment:ch1"

**Explanation**
- `library(readxl)` reads Excel files (Wickham H, Bryan J. readxl: Read Excel Files. R package version
  1.5.0.1, CRAN).
- `getGEO()` and `pData()` as in notebook 01.
- `grep(":ch1$", names(pd), value = TRUE)` lists the sample fields; `value = TRUE` returns their
  names rather than their positions.

## Match the columns of the authors' file to GSM identifiers

In [2]:
f <- raw("GSE232381", "GSE232381_Processed_data_files-Active_LN_vs_Inactive_LN.xlsx")
header <- names(read_excel(f, sheet = "Detail", n_max = 0))
label  <- sub("^(PAXgene-test-)?([^_]+)_.*$", "\\2", pd$description)
gsm_of <- setNames(rownames(pd), label)
measure <- sub(" [(].*$", "", header)
code    <- ifelse(grepl("[(].*[)]$", header), sub("^.*[(](.*)[)]$", "\\1", header), NA)
cols <- data.frame(position = seq_along(header), measure = measure,
                   gsm = ifelse(measure %in% c("Expected count", "TPM", "FPKM"), unname(gsm_of[code]), NA))
rm(header, label, gsm_of, code)
table(measure = cols$measure, matched_to_gsm = !is.na(cols$gsm))
stopifnot(all(!is.na(cols$gsm[cols$measure %in% c("Expected count", "TPM", "FPKM")])),
          all(table(cols$gsm[cols$measure == "Expected count"]) == 1))

                matched_to_gsm
measure          FALSE TRUE
  Description        1    0
  EC Number          1    0
  Expected count     0   25
  FPKM               0   25
  Fold change        1    0
  GO                 3    0
  GeneID             1    0
  PPEE               1    0
  Pathway            1    0
  TPM                0   25

**Explanation**
- `read_excel(f, sheet = "Detail", n_max = 0)` reads no data rows, only the column headers; `names()` gives
  them. The file has one sheet, "Detail".
- `label` is the first part of each GEO sample description, which is how the file labels its columns.
  `sub("^(PAXgene-test-)?([^_]+)_.*$", "\\2", ...)` keeps the text before the first `_`, after removing an
  optional `PAXgene-test-` prefix. For example, `"1234_HV7GTDSX3_L2"` gives `"1234"`.
- `gsm_of` maps each label to its GSM identifier.
- `measure` is the text of a header before the bracket, for example `"TPM"`; `code` is the text inside the
  brackets.
- `cols` keeps, for each column, its position, its measure and its GSM identifier. The labels themselves are
  removed with `rm()` and never printed.
- The table counts, per measure, the columns matched to a GSM identifier; `stopifnot()` stops the notebook
  unless every per-sample column is matched and each GSM identifier has exactly one "Expected count"
  column.

## Sample table

In [3]:
meta <- data.frame(sample = rownames(pd), ln_active = as.numeric(pd[["treatment:ch1"]] == "active LN"),
                   row.names = rownames(pd))
stopifnot(setequal(meta$sample, cols$gsm[cols$measure == "Expected count"]))
table(ln = ifelse(meta$ln_active == 1, "active", "inactive"))
saveRDS(meta, art("GSE232381", "metadata.rds"))
saveRDS(cols, art("GSE232381", "authors_columns.rds"))

ln
  active inactive 
      12       13 

**Explanation**
- `ln_active` is 1 for active lupus nephritis and 0 for inactive, from the GEO field `treatment`.
  `as.numeric(TRUE)` is 1.
- `setequal()` checks that the 25 GEO samples and the 25 "Expected count" columns are the same set.
- `saveRDS()` and `art()` as in notebook 01. `authors_columns.rds` holds positions, measures and GSM
  identifiers only.

**Result.** 25 samples: 12 active, 13 inactive. Every per-sample column of the authors' file matches one GSM
identifier.

**C0. This is not the paper's design.** Chen et al. (*Heliyon* 2024, Methods 3.5) sequenced 9 active and
9 inactive patients, "adjusted for age and sex (supplement Table 5)". GEO lists 25 samples and does not
say which 18 were compared.

**The paper's 18 sequenced patients** (supplement Table 5, "Comparison of laboratory parameters of systemic
lupus erythematosus patients with and without lupus nephritis (LN)"; mean ± SD; as printed):

| | Active LN (n = 9) | Inactive LN (n = 9) |
|---|---|---|
| Age | 17.11 ± 8.328 | 17.00 ± 4.243 |
| Sex | F:M = 8:1 | F:M = 7:2 |
| SLEDAI | 7.875 ± 4.203 | 8.160 ± 4.391 |
| anti-dsDNA (IU/mL) | 520.4 ± 302.8 **** | 306.0 ± 336.6 |
| C3 (mg/dL) | 79.95 ± 23.48 | 76.14 ± 25.64 |
| C4 (mg/dL) | 12.53 ± 5.854 **** | 17.82 ± 7.164 |
| WBC (K/µL) | 5.563 ± 3.031 ** | 6.511 ± 2.558 |
| Hb (g/dL) | 10.85 ± 1.695 | 11.22 ± 2.999 |
| Platelet (K/µL) | 291.3 ± 94.75 *** | 223.2 ± 88.08 |
| ESR (mm/hr) | 42.87 ± 31.57 **** | 18.87 ± 20.16 |
| CRP (mg/dL) | 0.3258 ± 0.6081 * | 0.1518 ± 0.2919 |
| Serum creatinine (mg/dL) | 0.5429 ± 0.1174 | 0.736 ± 0.3966 |
| eGFR (mL/min/1.73 m²) | 82.3 ± 21.73 | 80.52 ± 49.82 |

\*: P < 0.05; \*\*: P < 0.01; \*\*\*: P < 0.001; \*\*\*\*: P < 0.0001 (the table's footnote; the test is not named).
The table gives group summaries only, with no sample identifiers, so it cannot say which 18 GEO samples
were sequenced for the paper.

**C1. What "active" means here.** The paper defines active and inactive by renal outcome (Methods 3.2). Over all
episodes in the cohort, active episodes had a higher mean SLEDAI than inactive episodes (12.62 against 5.07;
Results 2.1, Table 2). In the 18 sequenced patients, mean SLEDAI is 7.875 (active) and 8.160 (inactive;
supplement Table 5). The paper
enrolled 95 patients with pediatric-onset SLE (Methods 3.1); GEO does not give each sequenced patient's age.